In [1]:
import pandas as pd
from sklearn.metrics import classification_report, accuracy_score, f1_score
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score
from imblearn.over_sampling import SMOTE
import os
import time

In [ ]:
# Constants
NUM_CLASSES = 4
RANDOM_STATE = 1082
MODEL_NAME = "Random Forest (SMOTE)"


In [ ]:
import sys as _sys
from pathlib import Path as _Path

_root = _Path.cwd().resolve()
while not (_root / "eval_metrics.py").is_file() and _root != _root.parent:
    _root = _root.parent
if str(_root) not in _sys.path:
    _sys.path.insert(0, str(_root))
from gps_ids import protocol


In [ ]:
EXP = protocol.smote_heldout("2d")
X_train_resampled = EXP.X_train
y_train_resampled = EXP.y_train
X_test_scaled = EXP.X_test
y_test = EXP.y_test


In [5]:
# Train the MLP Classifier model

model = model = RandomForestClassifier(
    n_estimators=200,
    n_jobs=-1,
    random_state=RANDOM_STATE
)

train_start_time = time.time()
model.fit(X_train_resampled, y_train_resampled)
train_time = time.time() - train_start_time

In [6]:
# Evaluate the model

print("\nEvaluating the model on the test set...")
y_pred = model.predict(X_test_scaled)
accuracy = accuracy_score(y_test, y_pred)
f1 = f1_score(y_test, y_pred, average='weighted', zero_division=0)
print(f"Model Accuracy on Test Set: {accuracy * 100:.2f}%")
print(f"F1 Score: {f1:.4f}")
print("\nClassification Report:")
print(classification_report(y_test, y_pred, zero_division=0))


Evaluating the model on the test set...
Model Accuracy on Test Set: 92.78%
F1 Score: 0.9308

Classification Report:
              precision    recall  f1-score   support

           0       0.98      0.94      0.96    318260
           1       0.76      0.87      0.81     29166
           2       0.67      0.81      0.73     35386
           3       0.93      0.97      0.95     25612

    accuracy                           0.93    408424
   macro avg       0.83      0.90      0.86    408424
weighted avg       0.94      0.93      0.93    408424



In [ ]:
from eval_metrics import evaluate, summary, time_inference
from gps_ids.results import append_validated

_proba = model.predict_proba(X_test_scaled) if hasattr(model, "predict_proba") else None
_timing = time_inference(model.predict, X_test_scaled)

row = evaluate(
    y_test,
    y_pred,
    model_name=MODEL_NAME,
    family="SMOTE",
    y_proba=_proba,
    train_time_sec=train_time,
    num_epochs="N/A",
    parameters=model.get_params(),
    timing=_timing,
    **EXP.eval_kwargs(),
)
append_validated(row, EXP.results_csv)
print(summary(row))
print(f"Results appended to {EXP.results_csv}")
